In [ ]:
from langgraph.graph import StateGraph, MessagesState,START, END,add_messages
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from typing import TypedDict,Annotated,List,Sequence
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from langgraph.prebuilt import ToolNode,tools_condition

In [ ]:
load_dotenv()

In [ ]:
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "practice-rag" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [ ]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

In [ ]:
class RAGState(TypedDict):
    message:Annotated[List[BaseMessage],add_messages]

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [ ]:

@tool
def rag_tool(query):
    '''A tool that takes a user query and a file path, retrieves relevant information from the file, and returns an answer based on the retrieved context.'''
    file_path = r"D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
    loader = PyPDFLoader(file_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)

    embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",output_dimensionality=384)

    vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

    rag_chain=RunnableParallel({
    "question" :RunnablePassthrough(),
    "context" :retriever| RunnableLambda(format_docs)
}
)
    parser=StrOutputParser()

    output=rag_chain | rag_prompt | llm | parser

    return {
        "query": query,
        "context": output
    }


In [ ]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [ ]:
def chat_rag(state:RAGState):
    response=llm_with_tools.invoke(state["message"])
    return {"message":[response]}

In [ ]:
tool_node=ToolNode([rag_tool])

In [ ]:
workflow=StateGraph(RAGState)
workflow.add_node("chat_rag",chat_rag)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "chat_rag")
workflow.add_conditional_edges(
    "chat_rag",
    tools_condition                      
)
workflow.add_edge("tools", "chat_rag")      

app=workflow.compile()

In [ ]:
result=app.invoke({
    "message":[HumanMessage(content=("What is aiml?"))]
    })